# Minecraft AI Bridge + vLLM TPU в Colab

[Пошаговый гайд для Windows](https://github.com/Cloudeah/minebridge/blob/main/docs/INGAME_VLLM_RU.md).

Модель **Gemma 4 E2B IT** работает на TPU в Colab. Minecraft, Python-контроллер и ngrok работают на Windows. Notebook получает наблюдение через HTTPS, вызывает локальный API vLLM и возвращает проверенное JSON-решение. Цели задаются в игре через **F7**, разрешение действий — **F8**, остановка — **F9**.

В Colab выбери **Среда выполнения → Сменить среду выполнения → TPU v5e-1**. Выполняй ячейки по порядку. Используется релиз `vllm-tpu==0.31.0` в отдельном окружении Python 3.12. Наличие TPU и совместимость конкретной среды проверяются при запуске; фактическая скорость пока не измерена разработчиком.

Ячейка 6 принимает до 30 решений для нескольких целей. После исчерпания этого лимита или 10 минут без целей повтори только её. Доступность TPU и время жизни среды зависят от [правил Colab](https://research.google.com/colaboratory/faq.html).


## 1. Установи vLLM TPU

Установка может занять несколько минут. Для релизного Linux x86_64 wheel нужен Python 3.12: `uv` создаёт отдельное окружение с этой версией, даже если Python ядра Colab отличается. Зависимости JAX/PyTorch устанавливаются в него. После сброса машины Colab повтори установку.

In [ ]:
import os
import sys
import subprocess
import time
import json
import base64
import signal
from pathlib import Path

VLLM_TPU_VERSION = "0.31.0"
VENV = Path("/content/minebridge-vllm-tpu")
VENV_PYTHON = VENV / "bin/python"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.0"], check=True)
# requests входит в стандартную среду Colab; её зависимости не обновляем в ядре.
import requests

install_environment = os.environ.copy()
install_environment["UV_CACHE_DIR"] = "/content/minebridge-uv-cache"
install_environment["UV_PYTHON_INSTALL_DIR"] = "/content/minebridge-python"
if not VENV_PYTHON.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VENV)],
                   env=install_environment, check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", str(VENV_PYTHON),
                f"vllm-tpu=={VLLM_TPU_VERSION}"], env=install_environment, check=True)
subprocess.run([str(VENV_PYTHON), "-c",
               "import sys, importlib.metadata as m; "
               "print('Python:', sys.version.split()[0]); "
               "print('vllm-tpu:', m.version('vllm-tpu')); "
               "print('tpu-inference:', m.version('tpu-inference'))"], check=True)

## 2. Проверь один TPU

Проверка выполняется в коротком отдельном процессе, который освобождает TPU после завершения. **Не импортируй JAX и не запускай `jax.devices()` в самом ядре notebook** перед сервером: другой процесс может занять чип. Если TPU недоступен, notebook остановится, а не переключится на CPU.

Эта конфигурация рассчитана на один чип v5e. Если уже запускал другую модель на TPU, освободи её или сбрось среду Colab.

JAX использует `tpu,cpu`: TPU выполняет модель, а CPU нужен загрузчику для подготовки весов. Проверка подтверждает доступность обоих backend до скачивания модели.

In [ ]:
server_environment = os.environ.copy()
server_environment.update({
    # CPU нужен tpu-inference для преобразования весов; TPU остаётся первым backend.
    "JAX_PLATFORMS": "tpu,cpu",
    "MODEL_IMPL_TYPE": "flax_nnx",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "HF_HOME": "/content/minebridge-hf-cache",
})
probe = """
import jax
devices = jax.devices('tpu')
print('JAX TPU devices:', devices, flush=True)
if len(devices) != 1 or any(device.platform != 'tpu' for device in devices):
    raise RuntimeError('This notebook requires exactly one TPU device (v5e-1).')
if jax.default_backend() != 'tpu':
    raise RuntimeError('TPU must be the default JAX backend for model execution.')
cpu_devices = jax.devices('cpu')
if not cpu_devices:
    raise RuntimeError('The vLLM TPU weight loader also requires a CPU backend.')
print('JAX CPU devices (weight loading):', cpu_devices, flush=True)
print('TPU device:', devices[0].device_kind, flush=True)
"""
subprocess.run([str(VENV_PYTHON), "-c", probe], env=server_environment, check=True, timeout=180)

## 3. Запусти сервер и скачай модель

По умолчанию используются текстовые наблюдения. Для первого запуска оставь `ENABLE_VISION = False`. Веса Gemma 4 E2B IT доступны публично; токен Hugging Face обычно не нужен.

Загрузка модели и первая компиляция могут занять несколько минут. API доступен только внутри Colab на `127.0.0.1:8000`; ngrok запускается на Windows. При ошибке смотри `/tmp/minebridge-vllm-tpu.log` в панели файлов Colab.

Для скриншотов позже поставь `ENABLE_VISION = True`, выполни ячейку остановки в конце notebook и повтори шаги 3–4. Один скриншот использует небольшой бюджет визуальных токенов. Меняя MODEL, выбирай модель с подтверждённой поддержкой TPU-плагина и подходящим размером.

In [ ]:
MODEL = "google/gemma-4-E2B-it"
MAX_MODEL_LEN = 3072
MAX_NEW_TOKENS = 256
ENABLE_VISION = False
IMAGE_TOKENS = 140
MODEL_TIMEOUT = 480
MODEL_URL = "http://127.0.0.1:8000"
LOG_PATH = Path("/tmp/minebridge-vllm-tpu.log")

def stop_vllm():
    process = globals().get("vllm_process")
    if process is not None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=10)
        globals()["vllm_process"] = None
    log = globals().get("vllm_log")
    if log is not None:
        log.close()
        globals()["vllm_log"] = None

settings = (MODEL, MAX_MODEL_LEN, ENABLE_VISION, IMAGE_TOKENS)
if not 512 <= MAX_MODEL_LEN <= 8192 or not 1 <= MAX_NEW_TOKENS < MAX_MODEL_LEN:
    raise ValueError("Проверь бюджет контекста и ответа")
previous = globals().get("vllm_process")
if previous is not None and previous.poll() is None:
    if globals().get("vllm_settings") != settings:
        raise RuntimeError("Настройки изменились. Сначала выполни ячейку остановки сервера.")
    print("Используем запущенный процесс vLLM.")
else:
    with requests.Session() as check_session:
        check_session.trust_env = False
        try:
            occupied = check_session.get(MODEL_URL + "/health", timeout=2)
        except requests.RequestException:
            occupied = None
        if occupied is not None:
            raise RuntimeError("Порт 8000 занят другим процессом. Освободи его или сбрось среду Colab.")
    command = [str(VENV / "bin/vllm"), "serve", MODEL,
               "--host", "127.0.0.1", "--port", "8000",
               "--dtype", "bfloat16", "--tensor-parallel-size", "1",
               "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", "1",
               "--max-num-batched-tokens", "512", "--enable-chunked-prefill",
               "--gpu-memory-utilization", "0.85"]
    if MODEL.startswith("google/gemma-4-"):
        command.append("--disable-chunked-mm-input")
        command.extend(["--limit-mm-per-prompt", json.dumps({
            "image": 1 if ENABLE_VISION else 0, "audio": 0, "video": 0,
        })])
        if ENABLE_VISION:
            command.extend(["--mm-processor-kwargs", json.dumps({"max_soft_tokens": IMAGE_TOKENS})])
    elif ENABLE_VISION:
        raise ValueError("Этот notebook настраивает изображения только для Gemma 4")
    vllm_log = LOG_PATH.open("ab")
    vllm_process = subprocess.Popen(command, env=server_environment, stdout=vllm_log,
                                    stderr=subprocess.STDOUT, start_new_session=True)
    vllm_settings = settings

old_model_session = globals().get("model_session")
if old_model_session is not None:
    old_model_session.close()
model_session = requests.Session()
model_session.trust_env = False
deadline = time.monotonic() + 1800
last_update = 0
try:
    while time.monotonic() < deadline:
        if vllm_process.poll() is not None:
            raise RuntimeError(f"vLLM завершился с кодом {vllm_process.returncode}; открой {LOG_PATH}")
        try:
            ready = model_session.get(MODEL_URL + "/v1/models", timeout=5)
            if ready.ok and any(item.get("id") == MODEL for item in ready.json().get("data", [])):
                print("vLLM готов, модель:", MODEL)
                break
        except (requests.RequestException, ValueError):
            pass
        if time.monotonic() - last_update >= 30:
            print("Ожидаем загрузку/компиляцию vLLM; журнал:", LOG_PATH, flush=True)
            last_update = time.monotonic()
        time.sleep(2)
    else:
        raise TimeoutError(f"vLLM не запустился за 30 минут; открой {LOG_PATH}")
except BaseException:
    stop_vllm()
    raise

## 4. Проверь API и JSON-ответ

Ячейка определяет адаптер и проверяет реальную генерацию перед подключением Minecraft. Она проверяет число токенов через `/tokenize`, преобразует PNG в формат OpenAI API и отклоняет обрезанные, неправильные или недопустимые решения.

Координаты должны быть уже вычисленными числами. При неверном JSON/решении адаптер выводит ответ модели и делает одну повторную попытку с уточнением формата. Всего допускаются две генерации в пределах `MODEL_TIMEOUT`; повтор также проверяется по длине контекста. Если обе попытки не прошли проверку, игровая задача не создаётся. Последний ответ можно посмотреть через `print(LAST_MODEL_REPLY)`.

`USE_STRUCTURED_OUTPUTS = False`: по умолчанию JSON-схема передаётся в инструкции, а ответ проверяется кодом. TPU-плагин отмечает строгую генерацию по схеме как непроверенную. Включать её следует после отдельной проверки поддержки в своей версии. Неправильный ответ не отправляется как игровое действие.

In [ ]:
USE_STRUCTURED_OUTPUTS = False
LAST_MODEL_REPLY = ""

def api_payload(payload):
    schema = payload["format"]
    messages = []
    for message in payload["messages"]:
        content = message["content"]
        images = message.get("images", [])
        if not isinstance(content, str) or not isinstance(images, list) or len(images) > 1:
            raise ValueError("Неверное сообщение или больше одного изображения")
        if images:
            if not ENABLE_VISION:
                raise ValueError("Включи ENABLE_VISION и перезапусти сервер для --vision")
            image = images[0]
            if not isinstance(image, str) or len(image) > 8 * 1024 * 1024:
                raise ValueError("Неверный размер PNG")
            if not base64.b64decode(image, validate=True).startswith(b"\x89PNG\r\n\x1a\n"):
                raise ValueError("Ожидался PNG")
            content = [{"type": "image_url", "image_url": {"url": "data:image/png;base64," + image}},
                       {"type": "text", "text": content}]
        if message["role"] == "system":
            content += ("\nReturn one JSON object only. Coordinates must be finite numbers already computed, "
                        "never expressions, variables or strings. Use available actions with documented effects; "
                        "choose wait if unsupported, finish only on confirmed success. "
                        "Keep reason under 120 characters. JSON schema: "
                        + json.dumps(schema, separators=(",", ":")))
        messages.append({"role": message["role"], "content": content})
    body = {"model": MODEL, "messages": messages, "stream": False,
            "temperature": 0, "max_tokens": MAX_NEW_TOKENS,
            "chat_template_kwargs": {"enable_thinking": False}}
    if USE_STRUCTURED_OUTPUTS:
        body["response_format"] = {"type": "json_schema", "json_schema": {
            "name": "minecraft_decision", "schema": schema,
        }}
    return body

def validate_item_parameters(action, parameters):
    from uuid import UUID
    import re
    if action == "item.drop":
        if parameters.get("hand", "main") != "main":
            raise ValueError("item.drop: only the selected main hand is supported")
        count = parameters.get("count", 1)
        if type(count) is not int or not 1 <= count <= 64:
            raise ValueError("item.drop: count must be an integer from 1 to 64")
        item = parameters.get("expected_item", "minecraft:air")
        if not isinstance(item, str) or not re.fullmatch(r"[a-z0-9_.-]+:[a-z0-9/._-]+", item):
            raise ValueError("item.drop: expected_item must be namespace:id")
    if action == "item.pickup":
        if "count" in parameters:
            raise ValueError("item.pickup: whole stack only; count is not supported")
        entity_id = parameters.get("entity_id")
        if not isinstance(entity_id, str):
            raise ValueError("item.pickup: entity_id must be an observed UUID")
        UUID(entity_id)
    if action == "item.use":
        if parameters.get("hand", "main") not in ("main", "off"):
            raise ValueError("item.use: hand must be main/off")
        ticks = parameters.get("duration_ticks", 40)
        if type(ticks) is not int or not 1 <= ticks <= 1200:
            raise ValueError("item.use: duration_ticks must be an integer from 1 to 1200")
        if "target" in parameters:
            target = parameters["target"]
            if not isinstance(target, dict) or any(type(target.get(axis)) not in (int, float) for axis in ("x", "y", "z")):
                raise ValueError("item.use: target must contain numeric x/y/z")
            json.dumps(target, allow_nan=False)


def parse_decision(response, schema):
    choice = response["choices"][0]
    content = choice["message"]["content"]
    global LAST_MODEL_REPLY
    LAST_MODEL_REPLY = content[:2000] if isinstance(content, str) else ""
    if choice.get("finish_reason") != "stop":
        raise ValueError("Ответ не завершён: уменьши цель/ответ или проверь модель")
    if not isinstance(content, str) or len(content.encode()) > 32768:
        raise ValueError("Неверный размер ответа модели")
    content = content.strip()
    if content.startswith("```json\n") and content.endswith("\n```"):
        content = content[8:-4].strip()
    def reject_constant(value):
        raise ValueError("Нефинитное число в решении")
    decision = json.loads(content, parse_constant=reject_constant)
    if not isinstance(decision, dict) or set(decision) != {"action", "parameters", "reason"}:
        raise ValueError("Нужны ровно action, parameters, reason")
    allowed = schema["properties"]["action"]["enum"]
    if not isinstance(decision["action"], str) or decision["action"] not in allowed:
        raise ValueError("Модель выбрала недоступное действие")
    if not isinstance(decision["parameters"], dict) or not isinstance(decision["reason"], str):
        raise ValueError("Неверные parameters/reason")
    if len(decision["reason"]) > 500:
        raise ValueError("Слишком длинный reason")
    json.dumps(decision, allow_nan=False)
    action = decision["action"]
    parameters = decision["parameters"]
    validate_item_parameters(action, parameters)

    def number(name, default, minimum, maximum):
        value = parameters.get(name, default)
        if type(value) not in (int, float) or not minimum <= value <= maximum:
            raise ValueError(f"{action}: {name} должен быть числом от {minimum} до {maximum}")
        return value

    if action == "move.jump":
        count = parameters.get("count", 1)
        if type(count) is not int or not 1 <= count <= 30:
            raise ValueError("move.jump: count должен быть целым числом от 1 до 30")
    if action == "move.crouch":
        ticks = parameters.get("ticks", 20)
        if type(ticks) is not int or not 1 <= ticks <= 200:
            raise ValueError("move.crouch: ticks должен быть целым числом от 1 до 200")
    if action in ("move.to", "move.relative"):
        for flag in ("sprint", "crouch"):
            if type(parameters.get(flag, False)) is not bool:
                raise ValueError(f"{action}: {flag} должен быть true или false")
        if parameters.get("sprint", False) and parameters.get("crouch", False):
            raise ValueError(f"{action}: sprint и crouch нельзя включать одновременно")
    if action == "move.relative":
        forward = number("forward", 0, -16, 16)
        right = number("right", 0, -16, 16)
        if forward == 0 and right == 0 or forward * forward + right * right > 16 * 16:
            raise ValueError("move.relative: расстояние должно быть больше 0 и не больше 16 блоков")
    if action == "look.turn":
        yaw = number("delta_yaw", 0, -180, 180)
        pitch = number("delta_pitch", 0, -90, 90)
        if yaw == 0 and pitch == 0:
            raise ValueError("look.turn: укажи ненулевой delta_yaw или delta_pitch")
    for name in ("target", "position", "origin"):
        if name in decision["parameters"]:
            vector = decision["parameters"][name]
            if not isinstance(vector, dict) or any(type(vector.get(axis)) not in (int, float)
                                                  for axis in ("x", "y", "z")):
                raise ValueError(f"Координаты {name}.x/y/z должны быть вычисленными числами")
    return decision

def infer_decision(payload):
    body = api_payload(payload)
    deadline = time.monotonic() + MODEL_TIMEOUT

    def request_timeout(maximum):
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            raise ValueError("Истекло время генерации корректного решения")
        return min(maximum, remaining)

    for attempt in range(2):
        token_request = {key: body[key] for key in
                         ("model", "messages", "chat_template_kwargs") if key in body}
        token_response = model_session.post(MODEL_URL + "/tokenize", json=token_request,
                                            timeout=request_timeout(120))
        token_response.raise_for_status()
        tokens = token_response.json()
        count = tokens["count"]
        limit = min(MAX_MODEL_LEN, tokens["max_model_len"])
        if not isinstance(count, int) or count < 0 or count + MAX_NEW_TOKENS > limit:
            raise ValueError(f"Контекст слишком длинный: {count} + {MAX_NEW_TOKENS} > {limit}. "
                             "Упрости цель или увеличь MAX_MODEL_LEN и перезапусти сервер.")
        response = model_session.post(MODEL_URL + "/v1/chat/completions", json=body,
                                      timeout=request_timeout(MODEL_TIMEOUT))
        response.raise_for_status()
        answer = response.json()
        if time.monotonic() >= deadline:
            raise ValueError("Истекло время генерации корректного решения")
        try:
            return parse_decision(answer, payload["format"])
        except ValueError as error:
            print(f"Некорректное решение, попытка {attempt + 1}/2: {error}", flush=True)
            print("Ответ модели:\n" + LAST_MODEL_REPLY, flush=True)
            if attempt == 1:
                raise ValueError(f"Модель не выдала корректное решение за 2 попытки: {error}") from error
            correction = ("\nYour previous reply failed validation: " + str(error)[:180]
                          + ". Generate a new decision from the same observation. "
                          "Return only valid JSON with exactly action, parameters and reason. "
                          "Compute coordinate numbers first; no expressions or variables, "
                          "even inside strings. Double-quote keys and strings; no trailing commas. "
                          "Use one available action and a short reason.")
            for message in body["messages"]:
                if message["role"] == "system":
                    message["content"] += correction
                    break
            else:
                body["messages"].insert(0, {"role": "system", "content": correction})

warmup_schema = {"type": "object", "additionalProperties": False,
                 "required": ["action", "parameters", "reason"], "properties": {
                     "action": {"type": "string", "enum": ["wait", "finish"]},
                     "parameters": {"type": "object"},
                     "reason": {"type": "string", "maxLength": 500},
                 }}
warmup_payload = {"format": warmup_schema, "messages": [
    {"role": "system", "content": "Return the requested JSON object. This is an API test."},
    {"role": "user", "content": 'Return {"action":"wait","parameters":{},"reason":"API ready"}'},
]}
print(json.dumps(infer_decision(warmup_payload), ensure_ascii=False, indent=2))

## 5. Подключись к Windows-контроллеру

Подготовь мод, Python и ngrok по [гайду](https://github.com/Cloudeah/minebridge/blob/main/docs/INGAME_VLLM_RU.md).

В первом PowerShell запусти контроллер для окна целей в игре:

```powershell
cd C:\minebridge
.\.venv\Scripts\python.exe examples\vllm_colab.py --config "$env:APPDATA\.minecraft\config\minebridge\bridge-owner.json" --interactive
```

Во втором PowerShell: `C:\ngrok\ngrok.exe http http://127.0.0.1:8766`.

Чтобы не вводить адрес и token каждый раз, открой **🔑 Секреты** в Colab, создай `MINEBRIDGE_RELAY_URL` с HTTPS-адресом ngrok и `MINEBRIDGE_RELAY_TOKEN` с **token из `.minebridge-colab.json`** и разреши notebook доступ к обоим. Если секрета нет или доступ выключен, ячейка попросит соответствующее значение. Token вводится скрыто и не выводится.

`bridge-owner.json` используется только на Windows. После изменения адреса ngrok обнови `MINEBRIDGE_RELAY_URL` и повтори эту ячейку. После изменения relay token обнови `MINEBRIDGE_RELAY_TOKEN`. Не записывай token в исходный код notebook.


In [ ]:
import getpass
from urllib.parse import urlsplit

def relay_setting(name, prompt, hidden=False):
    # Секреты необязательны: обычный Jupyter и Colab без доступа используют ввод.
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if isinstance(value, str) and value.strip():
        return value.strip()
    read = getpass.getpass if hidden else input
    return read(prompt).strip()

RELAY_URL = relay_setting("MINEBRIDGE_RELAY_URL", "HTTPS-адрес контроллера из ngrok: ").rstrip("/")
parsed_url = urlsplit(RELAY_URL)
if (parsed_url.scheme != "https" or not parsed_url.netloc or parsed_url.username
        or parsed_url.password or parsed_url.query or parsed_url.fragment or parsed_url.path):
    raise ValueError("Нужен HTTPS-адрес вида https://example.ngrok-free.app без пути")
RELAY_TOKEN = relay_setting("MINEBRIDGE_RELAY_TOKEN", "token из .minebridge-colab.json: ", hidden=True)
if len(RELAY_TOKEN) < 32:
    raise ValueError("Скопируй полный token из relay-файла")
old_session = globals().get("relay_session")
if old_session is not None:
    old_session.close()
relay_session = requests.Session()
relay_session.headers.update({"Authorization": "Bearer " + RELAY_TOKEN,
                              "ngrok-skip-browser-warning": "1"})
health = relay_session.get(RELAY_URL + "/health", timeout=30)
if health.status_code == 401:
    raise ValueError("Контроллер отклонил token. Проверь MINEBRIDGE_RELAY_TOKEN и token "
                     "из relay-файла, указанного при запуске Windows-контроллера; повтори шаг 5.")
health.raise_for_status()
if health.json().get("protocol") != "minebridge-colab-1":
    raise ValueError("Адрес не принадлежит совместимому контроллеру")
print("Контроллер:", health.json())


## 6. Принимай цели из Minecraft

По умолчанию ячейка принимает до **30 решений** для разных целей и продолжает ждать после завершения, отказа или остановки текущей цели. Запусти её, вернись в Minecraft, нажми **F8** и открой окно **F7**. Для первого задания выбери короткую цель, например «Посмотри на точку на три блока восточнее текущей позиции игрока».

**F9 немедленно останавливает управление мода**, даже когда модель думает. Новую цель можно задать в F7; ячейка продолжит работу. Когда появится сообщение об исчерпании лимита или пройдёт 10 минут без наблюдений, повтори только шаг 6. Лимит шагов одной цели задаётся в F7 отдельно; 30 здесь — общее число решений за один запуск ячейки.

Для скриншотов нужен сервер с `ENABLE_VISION = True`; затем включи зрение в F7. Отправляется PNG 320×180 вместе с наблюдением; карта глубины не передаётся этой модели.

Для отдельной команды из PowerShell вместо `--interactive` укажи `--goal "..." --steps 1` без `--execute` для просмотра решения. В этом режиме поставь `DECISIONS = 1`, `KEEP_READY_BETWEEN_GOALS = False`; выполнение включается флагом `--execute` и F8 в игре.


In [ ]:
def process_decisions(count=1, stop_on_wait=True):
    if type(count) is not int or not 1 <= count <= 30:
        raise ValueError("DECISIONS должен быть от 1 до 30")
    for index in range(count):
        deadline = time.monotonic() + 600
        job = None
        while time.monotonic() < deadline:
            try:
                response = relay_session.get(RELAY_URL + "/job", timeout=30)
                if response.status_code == 204:
                    continue
                response.raise_for_status()
                job = response.json()
                break
            except requests.RequestException:
                print("Контроллер недоступен. Проверь Windows/ngrok и запусти новую команду.")
                return
        if job is None:
            print("Наблюдение не пришло. Проверь Minecraft и локальный контроллер.")
            return
        print(f"Решение {index + 1}: наблюдение получено, модель {MODEL} думает...", flush=True)
        try:
            decision = infer_decision(job["payload"])
            result = {"job_id": job["job_id"], "decision": decision}
        except (requests.RequestException, ValueError, KeyError, IndexError, TypeError, RecursionError) as error:
            description = str(error)[:300] if isinstance(error, ValueError) else type(error).__name__
            result = {"job_id": job["job_id"], "error": "vLLM: " + description}
        # Повторяется доставка того же job_id, без новой генерации/игровой задачи.
        discarded = False
        for attempt in range(3):
            try:
                delivered = relay_session.post(RELAY_URL + "/result", json=result, timeout=30)
                if delivered.status_code == 409:
                    print("Задание уже истекло/остановлено. Решение не было принято.")
                    discarded = True
                    break
                delivered.raise_for_status()
                break
            except requests.RequestException:
                if attempt == 2:
                    print("Не удалось доставить ответ. Проверь Windows/ngrok; начни с нового наблюдения.")
                    return
                time.sleep(1)
        if discarded:
            if stop_on_wait:
                return
            continue
        if "error" in result:
            print(result["error"])
            if stop_on_wait:
                return
            continue
        print(json.dumps(decision, ensure_ascii=False, indent=2))
        if stop_on_wait and decision["action"] in ("wait", "finish"):
            return

    if not stop_on_wait:
        print("Лимит решений исчерпан. Повтори шаг 6, чтобы принимать новые цели из игры.")

DECISIONS = 30
KEEP_READY_BETWEEN_GOALS = True
process_decisions(DECISIONS, stop_on_wait=not KEEP_READY_BETWEEN_GOALS)

## 7. Останови сервер

Сначала **F9 в игре**, затем останови выполнение ячейки 6 и нажми **Ctrl+C в терминалах Windows-контроллера и ngrok**. Ячейка ниже останавливает только процесс vLLM, запущенный этим notebook, вместе с его дочерними процессами.

Для новой цели при работающем сервере используй F7; повторно выполнять установку и подключение не требуется. После изменения MODEL/ENABLE_VISION/MAX_MODEL_LEN останови сервер и повтори шаги 3–4. После перезапуска ngrok обнови подключение в шаге 5. Для полного завершения выбери **Среда выполнения → Отключиться и удалить среду выполнения**.

Локальные тесты репозитория проверяют адаптер и передачу решений через HTTP/WebSocket. Они не заменяют запуск нейросети и проверку производительности на реальном TPU.


In [ ]:
stop_vllm()
for name in ("model_session", "relay_session"):
    session = globals().get(name)
    if session is not None:
        session.close()
print("Сервер vLLM остановлен.")